# IPF File Preparation for DGE and Downstream Analysis

In [1]:
# load required packages
library(data.table)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   masks data.table::transpose()
✖ lubridate::wday() 

In [2]:
# read in data
counts <- fread("GSE213001_Entrez-IDs-Lung-IPF-GRCh38-p12-raw_counts.csv") #raw counts from GEO
meta <- fread("GSE213001_meta.txt", fill=TRUE) #series matrix file from GEO
emap <- fread("mart_export.txt", data.table=FALSE) #conversion between ensembl id and common gene symbol

In [3]:
head(meta)
head(counts)
head(emap)

!Sample_title,ALF008C,ALF013B,ALF004C,ALF019A,ALF030A,ALF042C,ALF040B,ALF004D,ALF020A,⋯,ALF023C,ALF033B,ALF048A,ALF010A,ALF027B,ALF012B,ALF007D,ALF025C,ALF042A,ALF032C
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
!Sample_geo_accession,GSM6568317,GSM6568318,GSM6568319,GSM6568320,GSM6568321,GSM6568322,GSM6568323,GSM6568324,GSM6568325,⋯,GSM6568446,GSM6568447,GSM6568448,GSM6568449,GSM6568450,GSM6568451,GSM6568452,GSM6568453,GSM6568454,GSM6568455
!Sample_status,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,⋯,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022,Public on Nov 02 2022
!Sample_submission_date,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,⋯,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022,Sep 09 2022
!Sample_last_update_date,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,⋯,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022,Nov 02 2022
!Sample_type,SRA,SRA,SRA,SRA,SRA,SRA,SRA,SRA,SRA,⋯,SRA,SRA,SRA,SRA,SRA,SRA,SRA,SRA,SRA,SRA
!Sample_channel_count,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1


V1,ALF001,ALF004C,ALF004D,ALF007A,ALF007B,ALF007C,ALF007D,ALF008A,ALF008B,⋯,ALF048A,ALF048B,ALF048C,ALF048D,ALF049A,ALF049B,ALF050A,ALF050B,ALF050C,ALF050D
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
ENSG00000279457,21,39,62,42,32,28,17,21,27,⋯,40,36,45,36,27,41,57,52,57,49
ENSG00000187634,287,379,383,313,188,402,158,413,777,⋯,71,46,33,253,60,111,42,18,23,28
ENSG00000188976,484,679,769,727,627,686,685,610,629,⋯,602,529,638,668,766,758,671,703,766,703
ENSG00000187961,66,228,230,198,113,181,75,156,186,⋯,98,138,147,87,142,152,237,183,264,195
ENSG00000187583,18,47,65,25,23,35,11,68,35,⋯,19,31,30,11,17,9,24,25,18,22
ENSG00000188290,142,173,216,181,168,179,264,106,250,⋯,143,194,150,206,180,121,224,139,140,94


,Gene stable ID,Gene name,Transcript stable ID
,<chr>,<chr>,<chr>
1,ENSG00000210049,MT-TF,ENST00000387314
2,ENSG00000211459,MT-RNR1,ENST00000389680
3,ENSG00000210077,MT-TV,ENST00000387342
4,ENSG00000210082,MT-RNR2,ENST00000387347
5,ENSG00000209082,MT-TL1,ENST00000386347
6,ENSG00000198888,MT-ND1,ENST00000361390


In [4]:
geneIDs <- counts$V1
counts$V1 <- NULL
counts <- as.matrix(counts)
rownames(counts) <- geneIDs

In [5]:
# counts: your matrix (genes x cells), rownames = Ensembl IDs
# emap: data.frame with columns 'gene' (Ensembl) and 'symbol'
is.numeric(counts)

# 1. Match Ensembl IDs to symbols
symbols <- emap$`Gene name`[match(rownames(counts), emap$`Gene stable ID`)]

# Optional: handle missing mappings
symbols[is.na(symbols)] <- rownames(counts)[is.na(symbols)]  # or drop them

# 2. Aggregate rows by symbol
counts_agg <- rowsum(counts, group = symbols)

# Done: rownames(counts_agg) are gene symbols
head(counts_agg)

[1] TRUE

,ALF001,ALF004C,ALF004D,ALF007A,ALF007B,ALF007C,ALF007D,ALF008A,ALF008B,ALF008C,⋯,ALF048A,ALF048B,ALF048C,ALF048D,ALF049A,ALF049B,ALF050A,ALF050B,ALF050C,ALF050D
,3550,1850,3096,2718,8412,9887,3258,3305,1558,1403,⋯,2194,2959,2055,1767,1775,2440,1463,1396,1772,1531
A2M,165871,80249,75169,181382,170068,142520,50222,162937,201332,195201,⋯,55966,66348,54162,43246,121380,118318,98106,83022,87190,97983
A4GALT,520,314,402,461,707,549,605,349,420,530,⋯,198,216,325,407,399,263,275,250,251,240
AAAS,340,575,709,556,495,615,304,445,529,512,⋯,320,380,426,216,499,671,505,500,578,515
AACS,181,400,493,232,345,218,622,231,279,333,⋯,244,235,253,438,293,372,406,370,391,371
AADAC,51,47,61,36,74,20,569,49,54,9,⋯,50,9,36,233,25,87,36,42,26,26


In [6]:
dim(counts_agg)
dim(counts)

[1] 15027   139

[1] 15065   139

In [8]:
# data contains IPF (disease) NDC (control) and some other conditions. check sample length for each
meta$`!Sample_title`
meta$ALF008C
sampledeets <- as.vector(t(meta[7]))
diseasegroup <- as.vector(t(meta[17]))
IPF <- c('diseasegroup: IPF', 'Lung,IPF.Apex,Whole lung', 'Lung,IPF.Base,Whole lung', 'Lung,IPF.NA,Whole lung')
ILD <- c('diseasegroup: ILD', 'Lung,ILD.Apex,Whole lung', 'Lung,ILD.Base,Whole lung')
NDC <- c('diseasegroup: NDC', 'Lung,NDC.Base,Whole lung', 'Lung,NDC.Apex,Whole lung', 'Lung,NDC.NA,Whole lung')
CLAD <- c('diseasegroup: CLAD', 'Lung,CLAD.Apex,Whole lung', 'Lung,CLAD.Base,Whole lung')

length(sampledeets[sampledeets %in% IPF == TRUE])
length(sampledeets[sampledeets %in% ILD == TRUE])
length(sampledeets[sampledeets %in% NDC == TRUE])
length(sampledeets[sampledeets %in% CLAD == TRUE])


[1] "!Sample_geo_accession"           "!Sample_status"                 
 [3] "!Sample_submission_date"         "!Sample_last_update_date"       
 [5] "!Sample_type"                    "!Sample_channel_count"          
 [7] "!Sample_source_name_ch1"         "!Sample_organism_ch1"           
 [9] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[11] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[13] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[15] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[17] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[19] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[21] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[23] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[25] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[27] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[29] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[31] "!Sample_characteristics_ch1"     "!Sample_characteristics_ch1"    
[33] "!Sample_characteristics_ch1"     "!Sample_molecule_ch1"           
[35] "!Sample_extract_protocol_ch1"    "!Sample_extract_protocol_ch1"   
[37] "!Sample_taxid_ch1"               "!Sample_data_processing"        
[39] "!Sample_data_processing"         "!Sample_data_processing"        
[41] "!Sample_platform_id"             "!Sample_contact_name"           
[43] "!Sample_contact_email"           "!Sample_contact_department"     
[45] "!Sample_contact_institute"       "!Sample_contact_address"        
[47] "!Sample_contact_city"            "!Sample_contact_state"          
[49] "!Sample_contact_zip/postal_code" "!Sample_contact_country"        
[51] "!Sample_data_row_count"          "!Sample_instrument_model"       
[53] "!Sample_library_selection"       "!Sample_library_source"         
[55] "!Sample_library_strategy"        "!Sample_relation"               
[57] "!Sample_supplementary_file_1"    "!series_matrix_table_begin"     
[59] "ID_REF"                          "!series_matrix_table_end"

[1] "GSM6568317"                                                                                                                                                                                                                                                                                                                                                         
 [2] "Public on Nov 02 2022"                                                                                                                                                                                                                                                                                                                                              
 [3] "Sep 09 2022"                                                                                                                                                                                                                                                                                                                                                        
 [4] "Nov 02 2022"                                                                                                                                                                                                                                                                                                                                                        
 [5] "SRA"                                                                                                                                                                                                                                                                                                                                                                
 [6] "1"                                                                                                                                                                                                                                                                                                                                                                  
 [7] "Lung,IPF.Apex,Whole lung"                                                                                                                                                                                                                                                                                                                                           
 [8] "Homo sapiens"                                                                                                                                                                                                                                                                                                                                                       
 [9] "tissue: Whole lung"                                                                                                                                                                                                                                                                                                                                                 
[10] "donorid: ALF008"                                                                                                                                                                                                                                                                                                                                                    
[11] "group: IPF.Apex"                                                                                                                                                                                                                                                                                                                                                    
[12] "li

[1] 62

[1] 26

[1] 41

[1] 10

In [9]:
dim(meta)

IPFmeta <- meta[, sampledeets %in% IPF]
NDCmeta <- meta[, sampledeets %in% NDC]

meta <- meta[7:34]
idx <- meta[[1]] == '!Sample_characteristics_ch1'

vals <- meta[[2]][idx]
newNames <- trimws(sub(":.*", "", vals)) #adjust metadata row/column names to be more descriptive
meta[[1]][idx] <- newNames

meta

[1]  60 140

!Sample_title,ALF008C,ALF013B,ALF004C,ALF019A,ALF030A,ALF042C,ALF040B,ALF004D,ALF020A,⋯,ALF023C,ALF033B,ALF048A,ALF010A,ALF027B,ALF012B,ALF007D,ALF025C,ALF042A,ALF032C
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
!Sample_source_name_ch1,"Lung,IPF.Apex,Whole lung","Lung,NDC.Base,Whole lung","Lung,ILD.Apex,Whole lung","Lung,IPF.Apex,Whole lung","Lung,NDC.Apex,Whole lung","Lung,ILD.Apex,Whole lung","Lung,IPF.Base,Whole lung","Lung,ILD.Base,Whole lung","Lung,NDC.Apex,Whole lung",⋯,"Lung,IPF.Apex,Whole lung","Lung,CLAD.Base,Whole lung","Lung,CLAD.Apex,Whole lung","Lung,NDC.Apex,Whole lung","Lung,IPF.Base,Whole lung","Lung,NDC.Base,Whole lung","Lung,ILD.Base,Whole lung","Lung,NDC.Apex,Whole lung","Lung,ILD.Apex,Whole lung","Lung,IPF.Apex,Whole lung"
!Sample_organism_ch1,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,⋯,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens,Homo sapiens
tissue,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,⋯,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung,tissue: Whole lung
donorid,donorid: ALF008,donorid: ALF013,donorid: ALF004,donorid: ALF019,donorid: ALF030,donorid: ALF042,donorid: ALF040,donorid: ALF004,donorid: ALF020,⋯,donorid: ALF023,donorid: ALF033,donorid: ALF048,donorid: ALF010,donorid: ALF027,donorid: ALF012,donorid: ALF007,donorid: ALF025,donorid: ALF042,donorid: ALF032
group,group: IPF.Apex,group: NDC.Base,group: ILD.Apex,group: IPF.Apex,group: NDC.Apex,group: ILD.Apex,group: IPF.Base,group: ILD.Base,group: NDC.Apex,⋯,group: IPF.Apex,group: CLAD.Base,group: CLAD.Apex,group: NDC.Apex,group: IPF.Base,group: NDC.Base,group: ILD.Base,group: NDC.Apex,group: ILD.Apex,group: IPF.Apex
lib.size,lib.size: 15573259,lib.size: 14146498,lib.size: 12253541,lib.size: 11850218,lib.size: 11836796,lib.size: 13033300,lib.size: 13232206,lib.size: 12033084,lib.size: 13188455,⋯,lib.size: 11812437,lib.size: 12348283,lib.size: 12753900,lib.size: 12276425,lib.size: 11558345,lib.size: 13007118,lib.size: 12503837,lib.size: 12017777,lib.size: 11212973,lib.size: 11266154
norm.factors,norm.factors: 1.00856304467136,norm.factors: 1.02594520742844,norm.factors: 0.978385212501405,norm.factors: 0.985785966310314,norm.factors: 0.992227170953119,norm.factors: 0.99321707994247,norm.factors: 1.14453734264326,norm.factors: 1.02693976721045,norm.factors: 0.987156782113402,⋯,norm.factors: 0.988917073252069,norm.factors: 1.14114597200271,norm.factors: 0.892434169046925,norm.factors: 0.885049569435388,norm.factors: 1.08873416705084,norm.factors: 0.882478583100834,norm.factors: 0.782919731362293,norm.factors: 1.04088750770238,norm.factors: 1.00125709880949,norm.factors: 1.09009337597802
lunglocation,lunglocation: Apex,lunglocation: Base,lunglocation: Apex,lunglocation: Apex,lunglocation: Apex,lunglocation: Apex,lunglocation: Base,lunglocation: Base,lunglocation: Apex,⋯,lunglocation: Apex,lunglocation: Base,lunglocation: Apex,lunglocation: Apex,lunglocation: Base,lunglocation: Base,lunglocation: Base,lunglocation: Apex,lunglocation: Apex,lunglocation: Apex
leftright,leftright: Right,leftright: Unclassified,leftright: Left,leftright: Left,leftright: Left,leftright: Right,leftright: Right,leftright: Left,leftright: Right,⋯,leftright: Right,leftright: Unclassified,leftright: Right,leftright: Right,leftright: Left,leftright: Right,leftright: Right,leftright: Left,leftright: Left,leftright: Right


In [10]:
rownames(meta) <- meta[[1]]
metaclean <- meta[, -1]

meta[idx, (colnames(meta)) := lapply(.SD, function(x) {
  trimws(sub("^[^:]+:\\s*", "", as.character(x)))
})]

In [11]:
metaclean <- t(meta)
colnames(metaclean) <- as.character(metaclean[1, ])
metaclean <- metaclean[-1, ]
head(metaclean)

,!Sample_source_name_ch1,!Sample_organism_ch1,tissue,donorid,group,lib.size,norm.factors,lunglocation,leftright,diseasenormal,⋯,processingdate,diseasesubtype,smokingstatus,severity,fvcprebd perc,dlco perc,dlco perc_corrected,diseasegroup1,age group,!Sample_molecule_ch1
ALF008C,"Lung,IPF.Apex,Whole lung",Homo sapiens,Whole lung,ALF008,IPF.Apex,15573259,1.00856304467136,Apex,Right,Disease,⋯,5/1/2018,IPF,Ex-Smoker,Severe,43,36,NA,IPF,Senior Adult (>=60),polyA RNA
ALF013B,"Lung,NDC.Base,Whole lung",Homo sapiens,Whole lung,ALF013,NDC.Base,14146498,1.02594520742844,Base,Unclassified,Normal,⋯,5/1/2018,NDC,Ex-Smoker,Unknown,NA,NA,NA,NDC,Adult [19-59],polyA RNA
ALF004C,"Lung,ILD.Apex,Whole lung",Homo sapiens,Whole lung,ALF004,ILD.Apex,12253541,0.978385212501405,Apex,Left,Disease,⋯,5/1/2018,NSIP,Ex-Smoker,Severe,38,NA,NA,ILD,Adult [19-59],polyA RNA
ALF019A,"Lung,IPF.Apex,Whole lung",Homo sapiens,Whole lung,ALF019,IPF.Apex,11850218,0.985785966310314,Apex,Left,Disease,⋯,5/1/2018,IPF,Never,Advanced,47,16,NA,IPF,Adult [19-59],polyA RNA
ALF030A,"Lung,NDC.Apex,Whole lung",Homo sapiens,Whole lung,ALF030,NDC.Apex,11836796,0.992227170953119,Apex,Left,Normal,⋯,5/2/2018,NDC,Unknown,Unknown,NA,NA,NA,NDC,Adult [19-59],polyA RNA
ALF042C,"Lung,ILD.Apex,Whole lung",Homo sapiens,Whole lung,ALF042,ILD.Apex,13033300,0.99321707994247,Apex,Right,Disease,⋯,5/8/2018,IPF,Never,Advanced,39,28,28,ILD,Senior Adult (>=60),polyA RNA


In [12]:
colnames(metaclean)
metaclean <- as.data.frame(metaclean)
head(metaclean$diseasegroup)

metaclean <- filter(metaclean, metaclean$diseasegroup %in% c("IPF", "NDC"))
unique(metaclean$diseasegroup)
dim(metaclean)

[1] "!Sample_source_name_ch1" "!Sample_organism_ch1"   
 [3] "tissue"                  "donorid"                
 [5] "group"                   "lib.size"               
 [7] "norm.factors"            "lunglocation"           
 [9] "leftright"               "diseasenormal"          
[11] "diseasegroup"            "gender"                 
[13] "diagnosis"               "age"                    
[15] "lungweight mg"           "rnaconcentration ngul"  
[17] "rin"                     "volum ul"               
[19] "processingdate"          "diseasesubtype"         
[21] "smokingstatus"           "severity"               
[23] "fvcprebd perc"           "dlco perc"              
[25] "dlco perc_corrected"     "diseasegroup1"          
[27] "age group"               "!Sample_molecule_ch1"

[1] "IPF" "NDC" "ILD" "IPF" "NDC" "ILD"

[1] "IPF" "NDC"

[1] 103  28

In [13]:
all(rownames(metaclean) %in% colnames(counts))
counts <- as.data.frame(counts)

counts <- counts %>% select(rownames(metaclean))
counts

[1] TRUE

,ALF008C,ALF013B,ALF019A,ALF030A,ALF040B,ALF020A,ALF047A,ALF029A,ALF032B,ALF037B,⋯,ALF028C,ALF035C,ALF034A,ALF044B,ALF023C,ALF010A,ALF027B,ALF012B,ALF025C,ALF032C
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
ENSG00000279457,29,31,43,49,26,20,53,34,53,50,⋯,5,53,44,46,39,16,67,35,49,45
ENSG00000187634,846,168,241,46,211,108,270,281,235,336,⋯,214,582,82,109,293,40,271,77,80,122
ENSG00000188976,774,1168,568,565,813,590,747,655,746,1011,⋯,781,673,597,761,640,519,650,826,668,610
ENSG00000187961,154,211,112,103,142,118,156,170,158,279,⋯,111,148,100,159,113,115,160,196,140,140
ENSG00000187583,63,42,19,11,41,26,25,34,21,72,⋯,49,20,6,27,26,5,57,45,9,47
ENSG00000188290,197,115,196,81,156,172,181,172,101,163,⋯,176,195,119,97,102,124,219,78,160,181
ENSG00000187608,480,299,337,172,118,361,319,595,344,353,⋯,175,750,81,336,209,338,138,1523,517,166
ENSG00000188157,2155,1016,1767,1735,2263,2396,2196,3019,1677,1985,⋯,2542,3122,1979,1777,1464,1046,2769,609,1803,2525
ENSG00000131591,198,156,140,155,203,158,218,173,168,204,⋯,128,157,143,242,132,127,200,211,115,180


In [14]:
# save metadata and filtered counts for DGE and downstream analysis
#fwrite(counts, row.names=TRUE, "GSE213001_filtered_counts.txt")
#fwrite(metaclean, row.names=TRUE, "GSE213001_filtered_metadata.txt")